In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
csv_path = "yolo_metrics.csv" # input validation results derived from k-fold methodology
class_order = ["Normal", "Blast"]
metrics = ["precision", "recall", "map50"]

error_type = "sd"

point_color = "#1C6082"
font_family = "Arial"

In [ ]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = [font_family]
plt.rcParams["font.size"] = 20
plt.rcParams["axes.titlesize"] = 22
plt.rcParams["axes.labelsize"] = 22
plt.rcParams["xtick.labelsize"] = 22
plt.rcParams["ytick.labelsize"] = 15

In [ ]:
df = pd.read_csv(csv_path)
df.head()

In [ ]:
required_cols = {"fold", "class", "precision", "recall", "map50"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

In [ ]:
df = df[df["class"].isin(class_order)].copy()
df

In [ ]:
summary = (
    df.groupby("class")[metrics]
    .agg(["mean", "std", "count"])
    .reindex(class_order)
)

summary

In [ ]:
x = np.arange(len(class_order))

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.8))

for ax, metric in zip(axes, metrics):
    means = summary[(metric, "mean")].to_numpy()
    stds = summary[(metric, "std")].fillna(0).to_numpy()
    counts = summary[(metric, "count")].to_numpy()

    ses = np.divide(
        stds,
        np.sqrt(counts),
        out=np.zeros_like(stds),
        where=counts > 0
    )

    errs = stds if error_type == "sd" else ses

    # error bars
    ax.errorbar(
        x,
        means,
        yerr=errs,
        fmt="none",
        ecolor=point_color,
        capsize=7,
        elinewidth=2,
        capthick=2,
        zorder=1
    )

    ax.set_xlim(-0.5, len(class_order) - 0.5)
    ax.set_ylim(0.5, 1)

    ax.set_xticks(x)
    ax.set_xticklabels(class_order, fontweight="bold")

    ylabel = "mAP50" if metric == "map50" else metric.capitalize()
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title("Model Performance", fontweight="bold", pad=10)

    ax.tick_params(axis="x", length=0, pad=8)
    ax.tick_params(axis="y", length=0)

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

In [ ]:
fig.subplots_adjust(left=0.07, right=0.98, bottom=0.18, top=0.86, wspace=0.38)

plt.savefig("fig_3b_performance_bars.png", dpi=300, bbox_inches="tight")
plt.show()